# AI4GSD Demo
------------------------------------------
## Smartphone photos
------------------------------------------
#### Table of contents
#### 0. Installing requried packages
#### 1. Importing required packages
#### 2. Defining working directory
#### 3. Extracting or copying your videos/photos to desired folder
#### 4. Obtaining scales from AI, drone flight data, or manual measurements
#### 5. Obtaining segmentation of anthropogenic objects
#### 6. Obtaining grain size data
#### 7. Applying quality control

##### *** Please contact Yunxiang Chen, yunxiang.chen@pnnl.gov for reporting errors

------------------------------------------
### 0. Installing requried packages

#### 0a. Downloading AI4GSD data from GitHub

In [7]:
from pathlib import Path
import sys, subprocess
WorkDir = Path("/content/AI4GSD")
if not WorkDir.exists(): subprocess.run(["git", "clone", "--progress", "https://github.com/pnnl/AI4GSD.git", str(WorkDir)],check=True)

### 0b. Install requried packages

In [8]:
%pip install -q numpy==2.2.6 pandas==2.2.3 matplotlib==3.10.6 opencv-python==4.12.0.88
%pip install -q torch torchvision ultralytics==8.3.186
%pip install -q natsort pyproj PyExifTool piexif imutils shapely psutil

### 0c. Define Google drive parement path and add it to system.

In [9]:
sys.path.insert(0, str(WorkDir))
print([item.name for item in WorkDir.iterdir()])

['.gitignore', 'AI4GSD', 'Demo_JupyterNotebook', 'Disclaimer Notice.txt', '.gitattributes', 'Install.md', '.git', 'Demo_Data', 'Demo_Python_Release', 'Licenses', 'Scales', 'README.md']


### 1. Importing required packages

In [10]:
import AI4GSD

### 2. Defining working directory

In [11]:
AI4GSD.pcinfo()
foldername = '20250517_YC_SmartphoneParkTest'                               # Parent folder where you save photos/videos.
PhotoDir = str(WorkDir / 'Demo_Data' / foldername)                         # AI4GSD use string path.

Machine name: 4df246fa1dca
Operating system: Ubuntu 24.04.5 LTS
Architecture: x86_64
Processor: Intel(R) Xeon(R) CPU @ 2.20GHz
Reported CPU frequency: 2.20 GHz


### 3. Extracting or copying your videos/photos to desired folder

In [12]:
AI4GSD.getPhotos(PhotoDir,nframe=2000, overwrite=False)                    # Use once.

### 4. Obtaining scales from AI, drone flight data, or manual measurements

In [13]:
PP = AI4GSD.parameters()
PP.Directory = PhotoDir
PP.OverWriteAll = True                                                     # Overwrite everything if True.
PP.OverWriteScaleFile = False
PP.OverWriteScaleRows = True
PP.ScaleSource = 'AI'                                                      # Use AI option.
PP.ModelName = 'YOLO11m.1280.20250825'                                     # Which AI to use.
PP.ComputingSource = 'local'                                               # What computing resource to use.
PP.ImageSizeFactor = 1                                                     # Do you need to user a larger image size for inference?
PP.ConfidenceThresholdUser = 0.85                                          # Minimum confidence threshold.
PP.YOLOSave = False                                                        # Need to save AI prediction to folder.
scales =  AI4GSD.metadata(PP)

Generating scale file for: 20250517_YC_SmartphoneParkTest
Estimating photo resolution via local AI model: YOLO11m.1280.20250825
Model layer: 113, model parmeter: 20.06M, submodule number: 409
IOU: 0.45, user confidence: 0.85
1/5: photo 1/5, IMG_9581.JPG, scale greencap, res 0.21 mm/px, status 200, (20.00%), 10.86 s
2/5: photo 2/5, IMG_9582.JPG, scale greencap, res 0.49 mm/px, status 200, (40.00%), 4.94 s
3/5: photo 3/5, IMG_9583.JPG, scale greencap, res 0.05 mm/px, status 200, (60.00%), 6.24 s
4/5: photo 4/5, IMG_9584.JPG, scale greencap, res 0.25 mm/px, status 200, (80.00%), 4.87 s
5/5: photo 5/5, IMG_9585.JPG, scale greencap, res 0.46 mm/px, status 200, (100.00%), 5.04 s
Total time: 32.30 s, time per photo 6.46 s
-----------------------------------------------------------------



### 5. Obtaining segmentation of anthropogenic objects

In [14]:
PP = AI4GSD.parameters()
PP.Directory = PhotoDir
PP.FolderLevel = 'L1'
PP.ModelName = 'YOLO11m.640.20250906'
PP.ObjectName = 'anthro'
PP.OverWriteSegementation = False
PP.YOLOSave = False
PP.YOLOIOU = 0.45
PP.ComputingSource = 'local'
PP.ImageSizeType = 'default'
PP.ImageSizeFactor=1
PP.SegmentationClassName = ['board','redbag','people']                     # What object class need to track.
AI4GSD.background(PP)

-----------------------------------------------------------------
Folder name: 20250517_YC_SmartphoneParkTest
output name: segments
Model name: YOLO11m.640.20250906 from local for anthro
Model layer: 125, model parmeter: 22.37M, submodule number: 445
IOU: 0.45, user confidence: 0.35
Number of photos: 5, number of exluded photos 0
20250517_YC_SmartphoneParkTest: 1/5, IMG_9581.JPG, mask existed, skipping... (20.00%)
20250517_YC_SmartphoneParkTest: 2/5, IMG_9582.JPG, mask existed, skipping... (40.00%)
20250517_YC_SmartphoneParkTest: 3/5, IMG_9583.JPG, mask existed, skipping... (60.00%)
20250517_YC_SmartphoneParkTest: 4/5, IMG_9584.JPG, mask existed, skipping... (80.00%)
20250517_YC_SmartphoneParkTest: 5/5, IMG_9585.JPG, mask existed, skipping... (100.00%)
Total time: 0.47 s, time per photo 0.09 s
-----------------------------------------------------------------



### 6. Obtaining grain size data

In [15]:
PP = AI4GSD.parameters()
PP.Directory = PhotoDir
PP.ModelName = 'YOLO11m.1280.20250322'
PP.ImageSizeFactor = 1
PP.LineThickness = 2
PP.YOLOSave = False
PP.SaveOverlayLabel = False
PP.OverWriteAll = True
PP.PrintOnScreen = True
PP.OverlayFontSize = 4
PP.OverlayLegendFontSize = 4
PP.OverlayGSDTickLineWidth = 4
gsd = AI4GSD.Photo2GSD(PP)

Output hidden; open in https://colab.research.google.com to view.

### 7. Applying quality control

In [16]:
gsdqc, stqc = AI4GSD.qualityControl(PhotoDir)


Quality control summary for 20250517_YC_SmartphoneParkTest:
Photos: time 31.0 s, area 6.8 m2, valid area 6.8 m2 (100.0%), frame 5, valid frame 5 (100.0%)
Photos: grain number 19139, number per image 3827.8; efficiency 0.2 m2/s, 617.4 grains/s
